# Model Evaluation: Confusion Matrix, ROC Curve, and AUC

## Importing Libraries

The following libraries are used to train the model, make predictions, and visualize its performance.

The `metrics` module from scikit-learn provides functions for computing ROC curves and AUC scores, while matplotlib is used for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score

## Loading and Preparing the Dataset

Missing values are handled, categorical variables are encoded, and the target variable is separated from the input features.

In [ ]:
train = pd.read_csv("titanic.csv")

X = train.drop("Survived", axis=1)
y = train["Survived"]

X["Deck"] = X["Cabin"].fillna("Unknown").str[0]
X = X.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

numerical_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Pclass", 
    "Deck"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])


## Creating a Train-Test Split and Fitting the Model

A single testing set is needed because we want one set of predictions from which we can build a confusion matrix and ROC curve.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

## Making Predictions

Two types of predictions are generated.

The first is the predicted class (survived or not survived).

The second is the predicted probability that each passenger survived. These probabilities are required to construct the ROC curve.

In [ ]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

## Building the Confusion Matrix Manually

A confusion matrix summarizes the performance of a binary classifier.

Instead of using scikit-learn's built-in confusion matrix function, the values are calculated manually to better understand what each quantity represents.

In [ ]:
TP = 0
TN = 0
FP = 0
FN = 0

for actual, predicted in zip(y_test, y_pred):

    if actual == 1 and predicted == 1:
        TP += 1

    elif actual == 0 and predicted == 0:
        TN += 1

    elif actual == 0 and predicted == 1:
        FP += 1

    elif actual == 1 and predicted == 0:
        FN += 1

Each prediction falls into one of four categories.

- **True Positive (TP):** predicted survived, actually survived.
- **True Negative (TN):** predicted did not survive, actually did not survive.
- **False Positive (FP):** predicted survived, but did not survive.
- **False Negative (FN):** predicted did not survive, but actually survived.

Together, these four values completely describe the classifier's predictions.

In [ ]:
print(f"TP = {TP}")
print(f"TN = {TN}")
print(f"FP = {FP}")
print(f"FN = {FN}")

## Computing Performance Metrics Manually

Using the confusion matrix, performance metrics can be calculated directly.

In [ ]:
accuracy = (TP + TN) / (TP + TN + FP + FN)

print(accuracy)

precision = TP / (TP + FP)

print(precision)

recall = TP / (TP + FN)

print(recall)

## Receiver Operating Characteristic (ROC) Curve

The confusion matrix evaluates the classifier at a single probability threshold (typically 0.5).

The ROC curve evaluates the classifier across every possible threshold.

For each threshold, it computes:

- True Positive Rate (Recall)
- False Positive Rate

Plotting these values illustrates the tradeoff between correctly identifying positive examples and producing false alarms.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,6))

plt.plot(fpr, tpr, label="Random Forest")

plt.plot([0,1],[0,1], linestyle="--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

The dashed diagonal line represents a random classifier.

A model whose ROC curve stays closer to the upper-left corner performs better because it achieves higher true positive rates while maintaining lower false positive rates.

## Area Under the Curve (AUC)

The Area Under the ROC Curve summarizes the ROC curve with a single number.

An AUC of:

- 0.5 indicates random guessing.
- 1.0 indicates perfect classification.

Higher AUC values indicate better overall discrimination between the two classes.

In [ ]:
auc = roc_auc_score(y_test, y_prob)

print(f"AUC = {auc:.3f}")

## Comparing Models with ROC and AUC

Using ROC and AUC, different models can be compared to see which model better classifies the data

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

lr_pred = logistic_model.predict(X_test)
lr_prob = logistic_model.predict_proba(X_test)[:, 1]

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)

lr_auc = roc_auc_score(y_test, lr_prob)



plt.figure(figsize=(7,7))

plt.plot(fpr, tpr,
         label=f"Random Forest (AUC = {auc:.3f})")

plt.plot(lr_fpr, lr_tpr,
         label=f"Logistic Regression (AUC = {lr_auc:.3f})")

plt.plot([0,1],[0,1],
         linestyle="--",
         color="gray",
         label="Random Guess")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.show()

print(f"Random Forest AUC      : {auc:.3f}")
print(f"Logistic Regression AUC: {lr_auc:.3f}")

### Overall, Random Forest classification is slightly better than logistic regression, but depending on priorities, such as getting a low false positive rate, logistic regression can be better.